In [ ]:
%pip install selenium
%pip install webdriver-manager
%pip install tqdm

## 모듈화한 함수 불러서 쓰기(각 링크 txt저장까지)

In [5]:
import sys
from pathlib import Path

# 현재 노트북 위치 (AC 폴더)
CURRENT_DIR = Path.cwd()

# 무조건 한 칸 위 (data_crawling)
BASE_DIR = CURRENT_DIR.parent

# import 경로 추가 (맨 앞에 넣는 게 더 안전)
sys.path.insert(0, str(BASE_DIR))

from selenium_auto_module import full_page, product_links

URL = "https://www.lge.co.kr/category/refrigerators"

html_sample = full_page(URL, "refrigerator")
print(html_sample)

links_sample = product_links("refrigerator")
print(links_sample)

페이지 로딩 완료
1번째 클릭
2번째 클릭
3번째 클릭
4번째 클릭
5번째 클릭
6번째 클릭
7번째 클릭
끝까지 펼침 완료
HTML 길이: 1529046
HTML 저장 완료: refrigerator_full_page_html.txt
<html lang="ko"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1"><link rel="preload" as="image" href="/kr/common/footer/sns_youtube.svg"><link rel="preload" as="image" href="/kr/common/footer/sns_insta.svg"><link rel="preload" as="image" href="/kr/common/footer/sns_facebook.svg"><link rel="preload" as="image" href="/kr/common/footer/sns_newsroom.svg"><link rel="preload" as="image" href="/kr/common/footer/sns_kakao.svg"><link rel="preload" as="image"
제품 카드 개수: 232
추출된 링크 개수: 232
저장 완료: refrigerator_product_links.txt
['https://www.lge.co.kr/refrigerators/w826svv492s__', 'https://www.lge.co.kr/refrigerators/m402nd', 'https://www.lge.co.kr/refrigerators/w826mqq492s__', 'https://www.lge.co.kr/refrigerators/j825gbb142__', 'https://www.lge.co.kr/refrigerators/m516svv012s']


## 재원추출 및 csv저장

In [2]:
import time
import re
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager


# =========================
# 1. 드라이버 실행
# =========================
driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install())
)

wait = WebDriverWait(driver, 10)


# =========================
# 2. 테스트용 링크 1개
# =========================
url = "https://www.lge.co.kr/refrigerators/t875mee111"


# =========================
# 공통 함수
# =========================
def clean_text(text):
    if not text:
        return ""
    return re.sub(r"\s+", " ", text).strip()


def get_product_name(soup):
    tag = soup.select_one("h1.name")
    if not tag:
        return ""

    for child in tag.select("span, em"):
        child.decompose()

    return clean_text(tag.get_text())


def get_image_url(soup):
    img = soup.select_one("#base_detail_target")
    if not img:
        return ""

    src = img.get("src", "").strip()
    if not src:
        return ""

    if src.startswith("http"):
        return src

    return "https://www.lge.co.kr" + src


def get_price(soup):
    tag = soup.select_one("small.original")
    if not tag:
        return ""

    for blind in tag.select(".blind"):
        blind.decompose()

    return clean_text(tag.get_text())


def get_spec(soup, *labels):
    spec_area = soup.select_one("#pdp_spec") or soup

    for dt in spec_area.find_all("dt"):
        dt_text = clean_text(dt.get_text(" ", strip=True))

        for label in labels:
            if label in dt_text:
                dd = dt.find_next_sibling("dd")
                if dd:
                    return clean_text(dd.get_text(" ", strip=True))

    return ""


def get_manual_pdf_url(soup):
    for top_area in soup.select("div.top-area"):
        strong = top_area.select_one("strong")
        title = clean_text(strong.get_text()) if strong else ""

        # 바로보기용 제외
        if title == "제품 사용 설명서":
            parent = top_area.find_parent()

            if parent:
                a_tag = parent.select_one("div.btn-group a[href]")

                if a_tag:
                    href = a_tag.get("href", "").strip()

                    if href.startswith("http"):
                        return href
                    elif href.startswith("/"):
                        return "https://www.lge.co.kr" + href

    return ""


# =========================
# 3. 실행
# =========================
driver.get(url)
time.sleep(3)

# 더보기 클릭
try:
    more_btn = wait.until(
        EC.element_to_be_clickable(
            (
                By.XPATH,
                "//a[contains(@title,'펼치기') or contains(., '제품 정보 더 보기')]"
            )
        )
    )

    driver.execute_script("arguments[0].click();", more_btn)
    time.sleep(2)

except:
    pass


soup = BeautifulSoup(driver.page_source, "html.parser")


# =========================
# 4. 데이터 추출
# =========================
result = {
    "제품명": get_product_name(soup),
    "이미지 링크": get_image_url(soup),
    "가격": get_price(soup),

    "에너지 소비효율등급": get_spec(soup, "에너지 소비효율등급", "에너지 소비효율등급"),
    "소비전력": get_spec(soup, "소비전력"),

    "설치타입": get_spec(soup, "설치타입"),
    "도어 개수": get_spec(soup, "도어 개수", "도어 수"),

    "전체 용량": get_spec(soup, "전체 용량"),
    "냉장 용량": get_spec(soup, "냉장 용량"),
    "냉동 용량": get_spec(soup, "냉동 용량"),

    "제품 크기": get_spec(soup, "제품 크기", "크기"),
    "무게": get_spec(soup, "무게"),

    "색상": get_spec(soup, "색상"),
    "도어 재질": get_spec(soup, "도어 재질", "도어스타일/재질"),

    "냉각방식": get_spec(soup, "냉각방식"),
    "스마트 진단": get_spec(soup, "스마트 진단"),
    "아이스메이커": get_spec(soup, "아이스메이커"),

    "제품 사용 설명서": get_manual_pdf_url(soup)
}


# =========================
# 5. 결과 출력
# =========================
for k, v in result.items():
    print(f"{k}: {v}")


# =========================
# 6. 종료
# =========================
driver.quit()

제품명: LG 디오스 오브제컬렉션 냉장고 (매직스페이스)
이미지 링크: https://www.lge.co.kr/kr/images/refrigerators/md10411859/gallery/medium-interior01.jpg
가격: 2,900,000원
에너지 소비효율등급: 1등급
소비전력: 43.7
설치타입: 프리스탠딩
도어 개수: 4도어
전체 용량: 870
냉장 용량: 503
냉동 용량: 367
제품 크기: 914 x 1,787 x 918
무게: 146
색상: 오브제컬렉션 베이지 / 베이지
도어 재질: 메탈
냉각방식: 멀티냉각방식
스마트 진단: O 있음
아이스메이커: 아이스 트레이
제품 사용 설명서: https://gscs-b2c.lge.com/open/downloadFile?fileId=LuQhmLf8ZMxg8pHShCqA


In [3]:
import time
import re
import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager


# =========================
# 1. 드라이버 실행
# =========================
driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install())
)

wait = WebDriverWait(driver, 10)


# =========================
# 2. 냉장고 상품 링크 txt 읽기
# =========================
with open("refrigerator_product_links.txt", "r", encoding="utf-8") as f:
    product_links = [line.strip() for line in f if line.strip()]

print("총 링크 개수:", len(product_links))


# =========================
# 3. 공통 함수
# =========================
def clean_text(text):
    if not text:
        return ""
    return re.sub(r"\s+", " ", text).strip()


def get_product_name(soup):
    tag = soup.select_one("h1.name")
    if not tag:
        return ""

    for child in tag.select("span, em"):
        child.decompose()

    return clean_text(tag.get_text())


def get_image_url(soup):
    img = soup.select_one("#base_detail_target")
    if not img:
        return ""

    src = img.get("src", "").strip()
    if not src:
        return ""

    if src.startswith("http"):
        return src

    return "https://www.lge.co.kr" + src


def get_price(soup):
    tag = soup.select_one("small.original")
    if not tag:
        return ""

    for blind in tag.select(".blind"):
        blind.decompose()

    return clean_text(tag.get_text())


def get_spec(soup, *labels):
    spec_area = soup.select_one("#pdp_spec") or soup

    for dt in spec_area.find_all("dt"):
        dt_text = clean_text(dt.get_text(" ", strip=True))

        for label in labels:
            if label in dt_text:
                dd = dt.find_next_sibling("dd")
                if dd:
                    return clean_text(dd.get_text(" ", strip=True))

    return ""


def get_manual_pdf_url(soup):
    for top_area in soup.select("div.top-area"):
        strong = top_area.select_one("strong")
        title = clean_text(strong.get_text()) if strong else ""

        # "제품 사용 설명서(바로보기용)" 제외
        if title == "제품 사용 설명서":
            parent = top_area.find_parent()

            if parent:
                a_tag = parent.select_one("div.btn-group a[href]")

                if a_tag:
                    href = a_tag.get("href", "").strip()

                    if href.startswith("http"):
                        return href
                    elif href.startswith("/"):
                        return "https://www.lge.co.kr" + href
                    else:
                        return href

    return ""


# =========================
# 4. 전체 링크 순회
# =========================
rows = []
failures = []

pbar = tqdm(product_links, desc="냉장고 상품 크롤링", ncols=120)

for idx, url in enumerate(pbar, start=1):
    try:
        pbar.set_postfix_str(f"{idx}/{len(product_links)} 접속 중")

        driver.get(url)
        time.sleep(3)

        # 제품 정보 / 제품 스펙 더 보기 클릭
        try:
            more_btn = wait.until(
                EC.element_to_be_clickable(
                    (
                        By.XPATH,
                        "//a[contains(@title,'펼치기') or contains(., '제품 정보 더 보기') or contains(., '제품스펙 상세정보 펼치기')]"
                    )
                )
            )

            driver.execute_script(
                "arguments[0].scrollIntoView({block:'center'});",
                more_btn
            )
            time.sleep(1)

            driver.execute_script("arguments[0].click();", more_btn)
            time.sleep(2)

        except:
            pass

        soup = BeautifulSoup(driver.page_source, "html.parser")

        row = {
            "제품명": get_product_name(soup),
            "이미지 링크": get_image_url(soup),
            "가격(원)": get_price(soup),

            "에너지 소비효율등급": get_spec(
                soup,
                "에너지소비효율등급",
                "에너지 소비효율등급",
                "에너지소비효율 등급",
                "에너지 소비효율 등급"
            ),

            "소비전력(W)": get_spec(
                soup,
                "소비전력",
                "소비 전력"
            ),

            "설치타입": get_spec(
                soup,
                "설치 타입",
                "설치타입",
                "타입"
            ),

            "도어 개수": get_spec(
                soup,
                "도어개수",
                "도어 개수",
                "도어수",
                "도어 수"
            ),

            "전체 용량(L)": get_spec(
                soup,
                "전체 용량",
                "전체용량"
            ),

            "냉장 용량(L)": get_spec(
                soup,
                "냉장용량",
                "냉장 용량"
            ),

            "냉동 용량(L)": get_spec(
                soup,
                "냉동용량",
                "냉동 용량"
            ),

            "제품 크기(mm)": get_spec(
                soup,
                "제품 크기",
                "제품크기",
                "크기 (WxHxD",
                "크기(WxHxD"
            ),

            "무게(kg)": get_spec(
                soup,
                "무게",
                "제품 무게",
                "제품무게"
            ),

            "색상": get_spec(
                soup,
                "색상",
                "컬러",
                "색깔"
            ),

            "도어 재질": get_spec(
                soup,
                "도어스타일/재질",
                "도어 스타일/재질",
                "도어 재질",
                "도어재질"
            ),

            "냉각방식": get_spec(
                soup,
                "냉각방식",
                "냉각 방식"
            ),

            "스마트 진단": get_spec(
                soup,
                "스마트진단",
                "스마트 진단"
            ),

            "아이스메이커": get_spec(
                soup,
                "아이스메이커",
                "아이스 메이커"
            ),

            "제품 사용 설명서": get_manual_pdf_url(soup),
        }

        rows.append(row)

        pbar.set_postfix_str(f"완료: {row['제품명'][:25]}")

        if idx % 10 == 0:
            pd.DataFrame(rows).to_csv(
                "lge_ref_temp.csv",
                index=False,
                encoding="utf-8-sig"
            )

    except Exception as e:
        failures.append({"url": url, "error": str(e)})
        pbar.set_postfix_str(f"실패: {idx}/{len(product_links)}")


# =========================
# 5. 최종 CSV 저장
# =========================
df = pd.DataFrame(rows)

df.to_csv(
    "lge_ref_all_products.csv",
    index=False,
    encoding="utf-8-sig"
)

print("전체 저장 완료: lge_ref_all_products.csv")
print("총 수집 개수:", len(df))
print("실패 개수:", len(failures))

display(df.head())


# =========================
# 6. 실패 목록 저장
# =========================
if failures:
    pd.DataFrame(failures).to_csv(
        "lge_ref_failures.csv",
        index=False,
        encoding="utf-8-sig"
    )
    print("실패 목록 저장 완료: lge_ref_failures.csv")


# =========================
# 7. 드라이버 종료
# =========================
driver.quit()

총 링크 개수: 232


냉장고 상품 크롤링: 100%|██████████████| 232/232 [1:19:07<00:00, 20.47s/it, 완료: LG 디오스 AI 오브제컬렉션 STEM 얼음정]2/232 접속 중]/s]

전체 저장 완료: lge_ref_all_products.csv
총 수집 개수: 232
실패 개수: 0


,제품명,이미지 링크,가격(원),에너지 소비효율등급,소비전력(W),설치타입,도어 개수,전체 용량(L),냉장 용량(L),냉동 용량(L),제품 크기(mm),무게(kg),색상,도어 재질,냉각방식,스마트 진단,아이스메이커,제품 사용 설명서
0,LG 트롬 오브제컬렉션 워시타워 + LG 디오스 AI 오브제컬렉션 냉장고 (매직스페이스),https://www.lge.co.kr/kr/images/wash-tower/md1...,"6,280,000원",세탁기 2등급 / 건조기 1등급,"3,500 (세탁 : 2,100 , 건조 : 1,400)",프리스탠딩,2도어,832,524,308,"700 x 1,890 x 830",전체 161.6 / 세탁 91 / 건조 70.6,네이처 베이지 / 네이처 그린,메탈,멀티냉각방식,O 있음,무빙 도어 아이스메이커,https://gscs-b2c.lge.com/open/downloadFile?fil...
1,LG 트롬 오브제컬렉션 워시타워 + LG 디오스 AI 오브제컬렉션 냉장고 (매직스페이스),https://www.lge.co.kr/kr/images/wash-tower/md1...,"5,780,000원",세탁기 2등급 / 건조기 1등급,"3,050 (세탁 : 2,100 , 건조 : 950)",프리스탠딩,2도어,832,524,308,"700 x 1,890 x 830",전체 162 / 세탁 91 / 건조 71,릴리 화이트 / 릴리 화이트,메탈,멀티냉각방식,O 있음,무빙 도어 아이스메이커,https://gscs-b2c.lge.com/open/downloadFile?fil...
2,LG 냉동고,https://www.lge.co.kr/kr/images/refrigerators/...,"700,000원",대상 외 제품,대상 외 제품,,,200,,,"600 x 1,383 x 629 mm",57,퓨어,,직냉식,,,https://gscs-b2c.lge.com/open/downloadFile?fil...
3,LG 일반냉장고,https://www.lge.co.kr/kr/images/refrigerators/...,"260,000원",4등급,10.2,,,43,43,,443 x 501 x 450,16,퓨어,,직냉식,,,https://gscs-b2c.lge.com/open/downloadFile?fil...
4,LG 일반냉장고,https://www.lge.co.kr/kr/images/refrigerators/...,"320,000원",4등급,19.4,,,90,90,,463 X 820 X 482,22,퓨어,,직냉식,,,
